# 原始半导体工业生产发布值与Stage1信号日

来源数值来自封存Fed G.17 Table1的同版NAICS3344变化。按冻结plan使用原生Qlib日历和保守的下个交易日可用钟，零市场结果、拟合或账户。

In [1]:
from pathlib import Path
import hashlib,json,os
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace missing")
    root=root.parent
card=root/"research/semiconductor_ip_stage1_availability_20261022"
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());source=root/plan['source_card']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized'] and not (card/'run_state.json').exists()
for f,k in [('summary.json','source_summary_sha256'),('verification_manifest.json','source_manifest_sha256'),('source_numeric_audit.json','source_numeric_audit_sha256'),('interpretation_note.json','source_interpretation_note_sha256')]:assert sha(source/f)==plan[k],f
assert sha(root/plan['Qlib_calendar_provider']/'qlib_store_manifest.json')==plan['Qlib_calendar_manifest_sha256']
atomic_json({'status':'source_calendar_attempted','plan_sha256':sha(plan_path),'source_object_load_attempts':1,'D_calendar_attempts':1,'DatasetH_attempts':1,'new_fits':0,'accounts':0},card/'run_state.json')
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
qlib.init(provider_uri=str(root/plan['Qlib_calendar_provider']),region='us',kernels=1)
objects,source_audit=load_verified_recorder_objects(source,expected_recorder_id=plan['source_recorder_id'],artifacts=[plan['source_artifact']],expected_manifest_sha256=plan['source_manifest_sha256'],expected_artifact_sha256={plan['source_artifact']:plan['source_artifact_sha256']})
events=objects[plan['source_artifact']].sort_values('release_date').reset_index(drop=True)
assert len(events)==192 and events.release_date.is_unique and events.ip_mom_pct_same_vintage.notna().all()
cal=pd.DatetimeIndex(D.calendar(start_time='2010-09-01',end_time='2026-10-07',freq='day')).normalize()
assert cal.is_unique and cal.is_monotonic_increasing
train=cal[(cal>=pd.Timestamp(plan['signal_bounds']['train'][0]))&(cal<=pd.Timestamp(plan['signal_bounds']['train'][1]))]
held=cal[(cal>=pd.Timestamp(plan['signal_bounds']['held_reused'][0]))&(cal<=pd.Timestamp(plan['signal_bounds']['held_reused'][1]))]
assert len(train)==plan['signal_bounds']['train_expected']==3020
assert len(held)==plan['signal_bounds']['held_expected']==944
signals=train.append(held)
releases=pd.DatetimeIndex(pd.to_datetime(events.release_date))
positions=cal.searchsorted(releases,side='right')
assert (positions<len(cal)).all()
available=cal[positions]
assert pd.DatetimeIndex(available).is_monotonic_increasing and not pd.DatetimeIndex(available).has_duplicates
lookup=available.searchsorted(signals,side='right')-1
assert (lookup>=0).all()
chosen=events.iloc[lookup].reset_index(drop=True)
release_for_signal=releases[lookup]
ages=np.asarray((signals-release_for_signal).days,dtype=int)
assert (ages>=1).all()
source_dates=pd.DatetimeIndex(pd.to_datetime(chosen.release_date))
assert (source_dates<signals).all()
full=pd.DataFrame({'signal_date':signals,'ip_mom_pct_same_vintage':chosen.ip_mom_pct_same_vintage.to_numpy(float),'release_age_days':ages,'source_release_date':chosen.release_date.to_numpy(),'source_observation_month':chosen.observation_month.to_numpy(),'source_index_base_year':chosen.index_base_year.to_numpy(int),'source_raw_sha256':chosen.raw_sha256.to_numpy(),'first_receipt_qualified':False})
assert len(full)==3964 and full.ip_mom_pct_same_vintage.notna().all()
feature=full.set_index('signal_date')[['ip_mom_pct_same_vintage','release_age_days']].copy()
feature['instrument']='SOXX_IP_NAICS3344';feature.index.name='datetime'
feature=feature.reset_index().set_index(['datetime','instrument'])
panel=pd.concat({'feature':feature},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={'all':(None,None)})
prepared=dataset.prepare('all',col_set=['feature'])
pd.testing.assert_frame_equal(prepared,panel)
years=full.signal_date.dt.year
annual=full.groupby(years).agg(signals=('signal_date','size'),mean_age_days=('release_age_days','mean'),maximum_age_days=('release_age_days','max'),distinct_releases=('source_release_date','nunique')).reset_index().rename(columns={'signal_date':'year'})
annual.to_csv(card/'yearly_availability.csv',index=False)
transitions=events.loc[events.index_base_year.ne(events.index_base_year.shift()),['release_date','index_base_year']]
audit={'status':'source_events_aligned_to_full_Stage1_signal_calendar_no_information_test','plan_sha256':sha(plan_path),'source_recorder_id':plan['source_recorder_id'],'native_Qlib_calendar_rows':len(cal),'train_signal_rows':len(train),'held_signal_rows':len(held),'mapped_source_rows':len(full),'missing_train_rows':0,'missing_held_rows':0,'first_train_source_release':full.iloc[0].source_release_date,'last_held_source_release':full.iloc[-1].source_release_date,'minimum_release_age_days':int(ages.min()),'maximum_release_age_days':int(ages.max()),'base_year_transitions':transitions.to_dict('records'),'historical_first_receipt_qualified':False,'same_day_release_use':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'Stage1_information_gate_tested':False,'SOXX_goal_achieved':False}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path='inputs',plan=plan,source_audit=source_audit)
    recorder.save_objects(artifact_path='sources',vintage_events=events,available_sessions=pd.DataFrame({'release_date':events.release_date,'available_session':available}))
    recorder.save_objects(artifact_path='analysis',mapped_signals=full,prepared_feature=prepared,availability_audit=audit)
    recorder.log_metrics(mapped_signals=len(full),new_fits=0,native_accounts=0)
    rid,eid=recorder.id,str(recorder.experiment_id)
artifacts=tracking/eid/rid/'artifacts';atomic_json({'recorder_id':rid,'entries':[{'path':str(p.relative_to(card)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
audit['recorder_id']=rid;audit['experiment_id']=eid;atomic_json(audit,card/'availability_audit.json')
binding=json.loads((source/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-07');atomic_json(binding,card/'nas_binding.json')
nas=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
summary={'experiment':plan['experiment'],'decision':'source_calendar_alignment_complete_information_and_PIT_unqualified','objective_achieved':False,'source_recorder_id':plan['source_recorder_id'],'result':'availability_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/eid/rid),'nas_evidence':nas,'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']}}
atomic_json(summary,card/'summary.json')
atomic_json({'status':'completed','plan_sha256':sha(plan_path),'source_object_loads':1,'D_calendar_calls':1,'DatasetH_builds':1,'recorder_id':rid,'new_fits':0,'accounts':0},card/'run_state.json')
(card/'conclusion.md').write_text(f"# 原始G.17工业生产来源对齐Stage1信号日\n\n使用原生Qlib日历一次与已封存来源Recorder，把192个原始发布版本保守延迟到下一真实交易日09:31才可用；只在该交易日收盘或以后作为Stage1信号日来源。2011–2022训练{len(train)}日、2023–2026已复用诊断{len(held)}日，均映射到当时最后一份发布版本，缺失0；最久来源年龄{int(ages.max())}日，逐年年龄见`yearly_availability.csv`。2007/2012/2017基年转换保留，未将绝对指数跨基年拼接。完整映射、版本哈希、DatasetH及所有日期在Qlib Recorder `{rid}`，NAS SHA `{nas['sha256']}`。\n\n本卡0市场结果、0拟合、0账户。发布日09:15为原文声明，采用下个真实交易日仅是保守可用代理，历史首次接收及跨基年统计语义仍未认证。决定：`source_calendar_alignment_complete_information_and_PIT_unqualified`。下一步冻结原Stage1 39项基线的匹配信息门，保持全部日历行和同源发布年龄；任何信息通过仍须完整0bp账户、2/5bp与独立前向。\n")
print(json.dumps({'status':summary['decision'],'train':len(train),'held':len(held),'max_age':int(ages.max()),'recorder_id':rid}))

{"status": "source_calendar_alignment_complete_information_and_PIT_unqualified", "train": 3020, "held": 944, "max_age": 78, "recorder_id": "9582d430c4b048b7b6623a99161ed71b"}


## 中文结论

以`conclusion.md`、`availability_audit.json`、`readback_verification.json`、逐年可用性小表及Qlib Recorder为准。保守下一交易日映射覆盖训练3020和诊断944信号日；2025最长年龄78天。来源对齐不证明历史真实首收、预测信息或费用后SOXX超额。